<a href="https://colab.research.google.com/github/jahnavimidde/Deep_learning/blob/main/DLLAB12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
text = "hello world"
chars = sorted(list(set(text)))
char2idx = {ch:i for i,ch in enumerate(chars)}
idx2char = {i:ch for ch,i in char2idx.items()}

data = [char2idx[c] for c in text]

seq_length = 4
X, Y = [], []
for i in range(len(data) - seq_length):
    X.append(data[i:i+seq_length])
    Y.append(data[i+seq_length])

X = torch.tensor(X)
Y = torch.tensor(Y)

class CharRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.rnn = nn.RNN(vocab_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

model = CharRNN(len(chars), 32)

def one_hot(x, vocab_size):
    return torch.nn.functional.one_hot(x, vocab_size).float()

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(300):
    inputs = one_hot(X, len(chars))
    outputs = model(inputs)
    loss = loss_fn(outputs, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

test = "hell"
test_idx = torch.tensor([[char2idx[c] for c in test]])
test_input = one_hot(test_idx, len(chars))

pred = model(test_input)
pred_char = idx2char[torch.argmax(pred).item()]

print("Input:", test)
print("Predicted next char:", pred_char)


Input: hell
Predicted next char: o


In [ ]:
import torch
import torch.nn as nn

sentence = "i love deep learning and i love ai"
words = sentence.split()

word2idx = {w:i for i,w in enumerate(set(words))}
idx2word = {i:w for w,i in word2idx.items()}

data = [word2idx[w] for w in words]

seq_length = 3
X, Y = [], []
for i in range(len(data) - seq_length):
    X.append(data[i:i+seq_length])
    Y.append(data[i+seq_length])

X = torch.tensor(X)
Y = torch.tensor(Y)

class WordLSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 16)
        self.lstm = nn.LSTM(16, 32, batch_first=True)
        self.fc = nn.Linear(32, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

model = WordLSTM(len(word2idx))

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(300):
    outputs = model(X)
    loss = loss_fn(outputs, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

test = ["i", "love", "deep"]
test_idx = torch.tensor([[word2idx[w] for w in test]])

pred = model(test_idx)
pred_word = idx2word[torch.argmax(pred).item()]

print("Input:", test)
print("Predicted next word:", pred_word)


Input: ['i', 'love', 'deep']
Predicted next word: learning


In [ ]:
import torch
import torch.nn as nn

sentence = "i love deep learning and i love ai"
words = sentence.split()

word2idx = {w:i for i,w in enumerate(set(words))}
idx2word = {i:w for w,i in word2idx.items()}

data = [word2idx[w] for w in words]
seq_length = 3
X, Y = [], []
for i in range(len(data) - seq_length):
    X.append(data[i:i+seq_length])
    Y.append(data[i+seq_length])

X = torch.tensor(X)
Y = torch.tensor(Y)
class WordLSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 16)
        self.lstm = nn.GRU(16, 32, batch_first=True)
        self.fc = nn.Linear(32, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

model = WordLSTM(len(word2idx))

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(300):
    outputs = model(X)
    loss = loss_fn(outputs, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
test = ["i", "love", "deep"]
test_idx = torch.tensor([[word2idx[w] for w in test]])

pred = model(test_idx)
pred_word = idx2word[torch.argmax(pred).item()]

print("Input:", test)
print("Predicted next word:", pred_word)


Input: ['i', 'love', 'deep']
Predicted next word: learning


In [ ]:
import torch
import torch.nn as nn

eng = ["hi", "hello"]
fra = ["salut", "bonjour"]
chars = list(set("".join(eng + fra)))
c2i = {c:i for i,c in enumerate(chars)}
i2c = {i:c for c,i in c2i.items()}

def encode(word):
    return torch.tensor([c2i[c] for c in word])
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(len(chars), 16)
        self.rnn = nn.LSTM(16, 32)

    def forward(self, x):
        x = self.embed(x)
        _, (h, c) = self.rnn(x)
        return h, c

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(len(chars), 16)
        self.rnn = nn.LSTM(16, 32)
        self.fc = nn.Linear(32, len(chars))

    def forward(self, x, h, c):
        x = self.embed(x).unsqueeze(0)
        out, (h, c) = self.rnn(x, (h, c))
        return self.fc(out.squeeze(0)), h, c

encoder = Encoder()
decoder = Decoder()

optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(500):
    total_loss = 0
    for e, f in zip(eng, fra):
        enc_input = encode(e)
        dec_input = encode(f)

        h, c = encoder(enc_input)

        loss = 0
        for i in range(len(dec_input)-1):
            out, h, c = decoder(dec_input[i], h, c)
            loss += loss_fn(out, dec_input[i+1])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
test = "hi"
h, c = encoder(encode(test))

pred = ""
inp = torch.tensor(c2i["s"])

for _ in range(5):
    out, h, c = decoder(inp, h, c)
    idx = torch.argmax(out).item()
    pred += i2c[idx]
    inp = torch.tensor(idx)

print("Input:", test)
print("Translated (approx):", pred)


Input: hi
Translated (approx): alutt
